1. Inspect user/item features
        
2. Convert categorical/numerical features
        
3. Build positive user-item pairs
        
4. Split train/test positives
        
5. Negative sampling
        
6. User Tower
        
7. Item Tower
        
8. Two-Tower + BCE
        
9. Precompute item embeddings
        
10. FAISS

11. Evaluating metrics
     
12. Filter already-interacted movies

In [111]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [112]:
# 1. Load User Demographics
user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

users = pd.read_csv(
    'ml-100k/u.user', 
    sep='|', 
    names=user_cols, 
    engine='python'
)

# 2. Load Item Features
genre_cols = [
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL'] + genre_cols

movie = pd.read_csv(
    'ml-100k/u.item', 
    sep='|', 
    names=movie_cols, 
    encoding='latin-1', 
    engine='python'
).drop(columns=['video_release_date', 'IMDb_URL'])

# 3. Load Ratings (u.data - tab-separated)
rating_cols = ['user_id', 'movie_id', 'rating', 'timestamp']

ratings = pd.read_csv(
    'ml-100k/u.data', 
    sep='\t', 
    names=rating_cols, 
    engine='python'
)

In [113]:
print(ratings.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [114]:
# Note that this move could break if IDs are not contiguous
ratings["movie_id"] = ratings['movie_id'] - 1 # For 0 idx
ratings["user_id"] = ratings['user_id'] - 1 # For 0 idx

In [115]:
num_users = ratings['user_id'].nunique()
num_items = ratings['movie_id'].nunique()

In [116]:
pos_df = ratings[ratings['rating']>=4]
pos_df['label']=1
pos_df = pos_df.drop(columns='rating')
pos_df.head()

,user_id,movie_id,timestamp,label
5,297,473,884182806,1
7,252,464,891628467,1
11,285,1013,879781125,1
12,199,221,876042340,1
16,121,386,879270459,1


In [117]:
pos_df.shape

(55375, 4)

In [118]:
# We;ll now split train_test by leave-one-out, specifically the last rating of each user
pos_df = pos_df.sort_values(["user_id", "timestamp"])
test_df = pos_df.groupby("user_id").tail(1)
train_df = pos_df.drop(test_df.index)
# notice that we don't even need sklearn train-test-split
# Our test cases will now = num_user because each user have only 1 test case. 
# We'll evaluate recall by how many users got recommeded their hid movie/ num_user

In [119]:
train_df = train_df.drop(columns='timestamp')
test_df = test_df.drop(columns='timestamp')

In [120]:
print(train_df.shape)
print(test_df.shape)

(54433, 3)
(942, 3)


## Deal with movies features

In [121]:
print(movie.head())

   movie_id        movie_title release_date  unknown  Action  Adventure  \
0         1   Toy Story (1995)  01-Jan-1995        0       0          0   
1         2   GoldenEye (1995)  01-Jan-1995        0       1          1   
2         3  Four Rooms (1995)  01-Jan-1995        0       0          0   
3         4  Get Shorty (1995)  01-Jan-1995        0       1          0   
4         5     Copycat (1995)  01-Jan-1995        0       0          0   

   Animation  Children's  Comedy  Crime  ...  Fantasy  Film-Noir  Horror  \
0          1           1       1      0  ...        0          0       0   
1          0           0       0      0  ...        0          0       0   
2          0           0       0      0  ...        0          0       0   
3          0           0       1      0  ...        0          0       0   
4          0           0       0      1  ...        0          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  
0        0        0        0 

In [122]:
movie['movie_id'] = movie['movie_id']-1 # for 0 idx
print(movie.head())

   movie_id        movie_title release_date  unknown  Action  Adventure  \
0         0   Toy Story (1995)  01-Jan-1995        0       0          0   
1         1   GoldenEye (1995)  01-Jan-1995        0       1          1   
2         2  Four Rooms (1995)  01-Jan-1995        0       0          0   
3         3  Get Shorty (1995)  01-Jan-1995        0       1          0   
4         4     Copycat (1995)  01-Jan-1995        0       0          0   

   Animation  Children's  Comedy  Crime  ...  Fantasy  Film-Noir  Horror  \
0          1           1       1      0  ...        0          0       0   
1          0           0       0      0  ...        0          0       0   
2          0           0       0      0  ...        0          0       0   
3          0           0       1      0  ...        0          0       0   
4          0           0       0      1  ...        0          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  
0        0        0        0 

In [123]:
#We'll add scaled release_year to movie_features for info richness

release_year = movie['release_date'].str.split('-').str[-1]
release_year[:5]

0    1995
1    1995
2    1995
3    1995
4    1995
Name: release_date, dtype: object

In [124]:
# convert to numeric, turning anything unparseable into NaN
release_year = pd.to_numeric(release_year, errors='coerce')

# check for missing values
print(release_year.isna().sum())

# fill missing with median year
release_year = release_year.fillna(release_year.median())

1


In [125]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
movie['release_year_scaled'] = scaler.fit_transform(release_year.values.reshape(-1, 1))
movie['release_year_scaled'][:5]

0    0.393842
1    0.393842
2    0.393842
3    0.393842
4    0.393842
Name: release_year_scaled, dtype: float64

We extract IDs since IDs is not an input of tower. But we don't throw away IDs since we need them to later find the movies in train_df

In [126]:
genre_cols = movie.columns[3:-2].tolist()

print(genre_cols)
print(len(genre_cols))

['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War']
18


In [127]:
movie_features = movie[genre_cols + ['release_year_scaled']].values
print(movie_features.shape)
print(movie_features[:5]) # first 5 movies' genre

(1682, 19)
[[0.         0.         0.         1.         1.         1.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.39384227]
 [0.         1.         1.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         1.         0.
  0.39384227]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         1.         0.
  0.39384227]
 [0.         1.         0.         0.         0.         1.
  0.         0.         1.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.39384227]
 [0.         0.         0.         0.         0.         0.
  1.         0.         1.         0.         0.         0.
  0.         0.         0.         0.         1.         0.
  0.39384227]]


## Move to user features

In [128]:
print(users.head())

   user_id  age gender  occupation zip_code
0        1   24      M  technician    85711
1        2   53      F       other    94043
2        3   23      M      writer    32067
3        4   24      M  technician    43537
4        5   33      F       other    15213


In [129]:
users['user_id'] = users['user_id']-1 # for 0 idx
print(users.head())

   user_id  age gender  occupation zip_code
0        0   24      M  technician    85711
1        1   53      F       other    94043
2        2   23      M      writer    32067
3        3   24      M  technician    43537
4        4   33      F       other    15213


In [130]:
# Encoding
users["gender_idx"] = users["gender"].map({"M": 0, "F": 1})

print(users[["gender", "gender_idx"]].head())

  gender  gender_idx
0      M           0
1      F           1
2      M           0
3      M           0
4      F           1


In [131]:
users = users.drop(columns='gender')
print(users.head())

   user_id  age  occupation zip_code  gender_idx
0        0   24  technician    85711           0
1        1   53       other    94043           1
2        2   23      writer    32067           0
3        3   24  technician    43537           0
4        4   33       other    15213           1


In [132]:
# Standard scaler on age

age_scaler = StandardScaler()

users["age"] = age_scaler.fit_transform(users[["age"]])

In [133]:
# Create a new variable then concat into users df is cleaner than directly create a new col in users df (users['occ_one_hot']=...)
occupation_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

occupation_encoded = occupation_encoder.fit_transform(
    users[["occupation"]]
)

occupation_cols = occupation_encoder.get_feature_names_out(["occupation"])

occupation_encoded_df = pd.DataFrame(
    occupation_encoded,
    columns=occupation_cols,
    index=users.index
)

users = pd.concat([users, occupation_encoded_df], axis=1)

In [134]:
users = users.drop(columns='occupation')

In [135]:
users.head()

,user_id,age,zip_code,gender_idx,occupation_administrator,occupation_artist,occupation_doctor,occupation_educator,occupation_engineer,occupation_entertainment,...,occupation_marketing,occupation_none,occupation_other,occupation_programmer,occupation_retired,occupation_salesman,occupation_scientist,occupation_student,occupation_technician,occupation_writer
0,0,-0.824859,85711,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,1.554867,94043,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,-0.906919,32067,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3,-0.824859,43537,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4,-0.086324,15213,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [136]:
# check to see how many unique zipcode are there
print(users['zip_code'].nunique(), "/", num_users)
# 795/943 means they're almost unique as IDs, which brings this model near a CF model
# 1 approach we can try is to seperate not by exact zipcode by regions

795 / 943


0	Connecticut, Massachusetts, Maine, New Hampshire, New Jersey, Puerto Rico, Rhode Island, Vermont, Virgin Islands

1	Delaware, New York, Pennsylvania

2	District of Columbia, Maryland, North Carolina, South Carolina, Virginia, West Virginia

3	Alabama, Florida, Georgia, plus Mississippi, Tennessee

4	Indiana, Kentucky, Michigan, Ohio

5	Iowa, Minnesota, Montana, N. Dakota, S. Dakota, Wisconsin

6	Illinois, Kansas, Missouri, Nebraska

7	(part of the central-south region — Texas/Oklahoma/Arkansas/Louisiana area)

8	Arizona, Colorado, Idaho, Nevada, New Mexico, Utah, Wyoming

9	California, Oregon, Washington, Alaska

Letter: Outside of US

In [ ]:
users['zip_region'] = users['zip_code'].astype(str).str[0]
users['zip_region'] = users['zip_region'].where(users['zip_region'].str.isdigit(), 'intl') 
print(users['zip_region'].value_counts()) 
# now this is much more informative than unique zipcode

zip_region
9       170
5       121
2       101
1        97
0        96
6        78
4        77
7        67
3        62
8        56
intl     18
Name: count, dtype: int64


In [138]:
zip_onehot = pd.get_dummies(users['zip_region'], prefix='zip_region',dtype=int) # One-hot zipcode 
print(zip_onehot.shape)  # should be (943, 11)

(943, 11)


In [139]:
users = users.drop(columns=["zip_code","zip_region"])
users.head()

,user_id,age,gender_idx,occupation_administrator,occupation_artist,occupation_doctor,occupation_educator,occupation_engineer,occupation_entertainment,occupation_executive,...,occupation_marketing,occupation_none,occupation_other,occupation_programmer,occupation_retired,occupation_salesman,occupation_scientist,occupation_student,occupation_technician,occupation_writer
0,0,-0.824859,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,1.554867,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,-0.906919,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3,-0.824859,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4,-0.086324,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We extract IDs since IDs is not an input of tower. But we don't throw away IDs since we need them to later find the users in train_df

In [140]:

# This line will go last when we done creating and modifying cols.
user_feature_cols = users.columns[1:].tolist()

print(user_feature_cols)
print(len(user_feature_cols))

['age', 'gender_idx', 'occupation_administrator', 'occupation_artist', 'occupation_doctor', 'occupation_educator', 'occupation_engineer', 'occupation_entertainment', 'occupation_executive', 'occupation_healthcare', 'occupation_homemaker', 'occupation_lawyer', 'occupation_librarian', 'occupation_marketing', 'occupation_none', 'occupation_other', 'occupation_programmer', 'occupation_retired', 'occupation_salesman', 'occupation_scientist', 'occupation_student', 'occupation_technician', 'occupation_writer']
23


In [141]:
user_features = np.hstack([
    users[user_feature_cols].values,        # your existing age + gender + occupation array
    zip_onehot.values
])
print(zip_onehot.columns)
print(user_features.shape)
print(user_features[:5])

Index(['zip_region_0', 'zip_region_1', 'zip_region_2', 'zip_region_3',
       'zip_region_4', 'zip_region_5', 'zip_region_6', 'zip_region_7',
       'zip_region_8', 'zip_region_9', 'zip_region_intl'],
      dtype='str')
(943, 34)
[[-0.82485939  0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          1.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          1.          0.          0.        ]
 [ 1.55486734  1.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          1.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          1.          0.        ]
 [-0.906

# Negative Sampling

In [142]:
user_positive_items = (
    pos_df
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)
user_positive_items[10]

{7,
 8,
 14,
 21,
 27,
 46,
 50,
 55,
 69,
 78,
 82,
 85,
 96,
 99,
 106,
 110,
 124,
 134,
 172,
 184,
 190,
 193,
 195,
 202,
 207,
 212,
 228,
 229,
 236,
 238,
 240,
 257,
 267,
 276,
 285,
 290,
 300,
 311,
 316,
 317,
 331,
 349,
 355,
 356,
 371,
 392,
 401,
 422,
 424,
 426,
 427,
 428,
 432,
 433,
 434,
 507,
 523,
 526,
 543,
 548,
 579,
 602,
 651,
 658,
 662,
 689,
 691,
 698,
 706,
 712,
 713,
 717,
 722,
 728,
 730,
 732,
 735,
 736,
 739,
 740,
 743,
 744,
 745,
 748,
 749,
 751}

In [143]:
user_disliked_items = (
    ratings[ratings['rating'] <= 3]
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

In [144]:
import random

train_row = []

for _, row in train_df.iterrows():
    user_id = row['user_id']
    positive_item_id = row['movie_id']

    # positive
    train_row.append([user_id, positive_item_id, 1])

    user_positive_items_set = user_positive_items[user_id]
    negative_candidates = list(set(range(num_items)) - user_positive_items_set)

    disliked_set = user_disliked_items.get(user_id, set())

    if disliked_set:
        # true negative
        train_row.append([user_id, random.choice(list(disliked_set)), 0])
        # unrated negative
        train_row.append([user_id, random.choice(negative_candidates), 0])
    else:
        # no 1-3 ratings available -> both negatives fall back to unrated
        train_row.append([user_id, random.choice(negative_candidates), 0])
        train_row.append([user_id, random.choice(negative_candidates), 0])

train_df = pd.DataFrame(train_row, columns=['user_id', 'movie_id', 'label'])

In [145]:
# Final check before plug this into Dataloader
print(train_df.head())
print(train_df["user_id"].min(), train_df["user_id"].max())
print(train_df["movie_id"].min(), train_df["movie_id"].max())

print(user_features.shape)
print(movie_features.shape)

   user_id  movie_id  label
0        0       167      1
1        0        72      0
2        0      1320      0
3        0       171      1
4        0       119      0
0 942
0 1681
(943, 34)
(1682, 19)


In [146]:
# Final check before plug this into Dataloader
row = train_df.iloc[0]

print(user_features[row["user_id"]])
print(movie_features[row["movie_id"]])
print(row["label"])

[-0.82485939  0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          1.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          1.          0.          0.        ]
[ 0.         0.         0.         0.         0.         1.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
 -1.0802805]
1


In [147]:
train_df.head()

,user_id,movie_id,label
0,0,167,1
1,0,72,0
2,0,1320,0
3,0,171,1
4,0,119,0


## We'll now plug data into dataloader

In [148]:

class CBFDataset(Dataset):
    def __init__(self, df, user_features, movie_features):
        self.user_ids = torch.tensor(
            df["user_id"].values,
            dtype=torch.long
        )

        self.movie_ids = torch.tensor(
            df["movie_id"].values,
            dtype=torch.long
        )

        self.labels = torch.tensor(
            df["label"].values,
            dtype=torch.float32
        )

        self.user_features = torch.tensor(
            user_features,
            dtype=torch.float32
        )

        self.movie_features = torch.tensor(
            movie_features,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        movie_id = self.movie_ids[idx]

        # Note that the order of this return needs to match the order in train loop
        return (
            self.user_features[user_id],
            self.movie_features[movie_id],
            self.labels[idx]
        )

In [149]:
train_set = CBFDataset(train_df,user_features,movie_features)
train_loader = DataLoader(train_set,256,shuffle=True)
# Because we don't plug test_df into MLP, that's why we don't need it to go through dataset and dataloader

In [150]:
user_x, movie_x, y = next(iter(train_loader))

print(user_x.shape)
print(movie_x.shape)
print(y.shape)

torch.Size([256, 34])
torch.Size([256, 19])
torch.Size([256])


## Build Two Tower

In [151]:
class TwoTower(nn.Module):
    def __init__(self, user_input_dim, movie_input_dim, embedding_dim=32):
        super().__init__()

        self.user_mlp = nn.Sequential(nn.Linear(user_input_dim, 64),
                                    nn.ReLU(),
                                    nn.Linear(64, embedding_dim))

        self.movie_mlp = nn.Sequential(nn.Linear(movie_input_dim, 64),
                                    nn.ReLU(),
                                    nn.Linear(64, embedding_dim))

    def forward(self,user_features,movie_features):
        user_embedding = self.user_mlp(user_features)
        movie_embedding = self.movie_mlp(movie_features)

        score = (user_embedding*movie_embedding).sum(dim=1)
        return score


In [152]:
# For OOP clarity, we're just building the MLP by providing the cols of user_features and movie_features, we haven't plug anything into forward for it to compute yet
# We will plug user_features ad movie_features for it to compute later when we model(user_features,model_features)
model = TwoTower(user_input_dim=user_features.shape[1],movie_input_dim=movie_features.shape[1],embedding_dim=32)

## Set loss, optim, device

In [153]:

device = torch.device(
    "mps" if torch.mps.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

print(device)

mps


## Train loop

In [154]:
epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0
    # Note that the order of this train loop needs to match the order in CBFDataset class
    for batch_user_features, batch_item_features, batch_labels in train_loader:

        batch_user_features = batch_user_features.to(device)
        batch_item_features = batch_item_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        logits = model(batch_user_features, batch_item_features)

        loss = criterion(logits, batch_labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Train Loss: {avg_loss:.4f}"
    )

Epoch 1/20, Train Loss: 0.6036
Epoch 2/20, Train Loss: 0.5892
Epoch 3/20, Train Loss: 0.5835
Epoch 4/20, Train Loss: 0.5794
Epoch 5/20, Train Loss: 0.5769
Epoch 6/20, Train Loss: 0.5737
Epoch 7/20, Train Loss: 0.5718
Epoch 8/20, Train Loss: 0.5702
Epoch 9/20, Train Loss: 0.5688
Epoch 10/20, Train Loss: 0.5674
Epoch 11/20, Train Loss: 0.5656
Epoch 12/20, Train Loss: 0.5644
Epoch 13/20, Train Loss: 0.5634
Epoch 14/20, Train Loss: 0.5625
Epoch 15/20, Train Loss: 0.5615
Epoch 16/20, Train Loss: 0.5605
Epoch 17/20, Train Loss: 0.5596
Epoch 18/20, Train Loss: 0.5588
Epoch 19/20, Train Loss: 0.5582
Epoch 20/20, Train Loss: 0.5571


In [155]:
model.eval() # Turn off training mode

with torch.no_grad():

    # Generate movie embeddings
    movie_tensor = torch.tensor(
        movie_features,
        dtype=torch.float32
    ).to(device)

    movie_embeddings = model.movie_mlp(movie_tensor)

    # Generate user embeddings
    user_tensor = torch.tensor(
        user_features,
        dtype=torch.float32
    ).to(device)

    user_embeddings = model.user_mlp(user_tensor)


# Convert embeddings to NumPy
movie_embeddings_np = (
    movie_embeddings
    .cpu()
    .numpy()
    .astype("float32")
)

user_embeddings_np = (
    user_embeddings
    .cpu()
    .numpy()
    .astype("float32")
)


# Build FAISS index
import faiss

embedding_dim = movie_embeddings_np.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(movie_embeddings_np)


# Make sure the query is contiguous float32
user_embeddings_np = np.ascontiguousarray(
    user_embeddings_np,
    dtype=np.float32
)

# Use a single CPU thread
faiss.omp_set_num_threads(1)

#For each user, we recommend all movies in score descending order. 
# Why? Because we'll take top_k later when we evaluate.
# If we take top_k in FAISS and then filter out interated movies in train_df, our top 10 could only have 2 left
# [A, B, C, D, E, F, G, H, I, J] but A-H are interacted movies, which means we're left with I,J after we filter our interacted movies
distances, movie_ids = index.search(
    user_embeddings_np,
    num_items
)

# So the matrix should be (943,1682). The order of rows start from user 0,1,2,3,...
print(movie_ids.shape) 
print(movie_ids[0]) # User 0 ALL movies in score descending order

(943, 1682)
[  49  188  407 ... 1291 1075 1372]


In [156]:
import numpy as np

K_values = [10, 50, 100, 200, 500]

# Recompute all_top_k for the LARGEST K once, then slice smaller Ks from it
max_K = max(K_values)

# ratings includes EVERY interaction (rating 1-5), including each user's held-out
# test movie (test_df is a subset of pos_df, which is a subset of ratings).
# If we filter using seen_movies_by_user as-is, the test movie can never appear
# in all_top_k, so recall@K would be 0 by construction, not by model performance.
# Fix: remove each user's held-out test movie from their "seen" set so it stays
# eligible to be retrieved.
test_movie_by_user = test_df.set_index("user_id")["movie_id"].to_dict()

seen_movies_by_user = (
    ratings
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

for user_id, test_movie_id in test_movie_by_user.items():
    seen_movies_by_user[user_id].discard(test_movie_id)

all_top_k = []
for user_id in range(num_users):
    ranked_movies = movie_ids[user_id]
    seen_movies = seen_movies_by_user.get(user_id, set())
    filtered_movies = [m for m in ranked_movies if m not in seen_movies]
    all_top_k.append(filtered_movies[:max_K])

results = {}

for K in K_values:
    hits = 0
    precision_sum = 0
    ndcg_sum = 0
    num_eval_users = 0

    for user_id in range(num_users):
        test_movie_id = test_movie_by_user.get(user_id)
        if test_movie_id is None:
            continue

        num_eval_users += 1
        top_k = all_top_k[user_id][:K]

        if test_movie_id in top_k:
            hits += 1
            rank = top_k.index(test_movie_id)
            precision_sum += 1 / K
            ndcg_sum += 1 / np.log2(rank + 2)

    results[K] = {
        "recall": hits / num_eval_users,
        "precision": precision_sum / num_eval_users,
        "ndcg": ndcg_sum / num_eval_users,
    }

for K, metrics in results.items():
    print(f"K={K:>4} | Recall: {metrics['recall']:.4f} | "
        f"Precision: {metrics['precision']:.4f} | NDCG: {metrics['ndcg']:.4f}")

K=  10 | Recall: 0.0403 | Precision: 0.0040 | NDCG: 0.0215
K=  50 | Recall: 0.1454 | Precision: 0.0029 | NDCG: 0.0441
K= 100 | Recall: 0.2187 | Precision: 0.0022 | NDCG: 0.0560
K= 200 | Recall: 0.3450 | Precision: 0.0017 | NDCG: 0.0736
K= 500 | Recall: 0.5786 | Precision: 0.0012 | NDCG: 0.1015


In [157]:
all_top_k[0] 
# This give us top k NON-INTERACTED (not even interaction where rating 1,2,3,4,5) movies of user 0. 
# But still have the held-out test case since we wanna see if it can retrieve it

[np.int64(407),
 np.int64(1005),
 np.int64(317),
 np.int64(312),
 np.int64(1239),
 np.int64(678),
 np.int64(1482),
 np.int64(497),
 np.int64(301),
 np.int64(802),
 np.int64(1366),
 np.int64(514),
 np.int64(325),
 np.int64(630),
 np.int64(430),
 np.int64(918),
 np.int64(421),
 np.int64(1438),
 np.int64(1193),
 np.int64(738),
 np.int64(582),
 np.int64(769),
 np.int64(513),
 np.int64(473),
 np.int64(434),
 np.int64(1518),
 np.int64(654),
 np.int64(1461),
 np.int64(543),
 np.int64(576),
 np.int64(529),
 np.int64(1422),
 np.int64(650),
 np.int64(1484),
 np.int64(285),
 np.int64(548),
 np.int64(1637),
 np.int64(1504),
 np.int64(1155),
 np.int64(627),
 np.int64(1567),
 np.int64(764),
 np.int64(708),
 np.int64(1018),
 np.int64(727),
 np.int64(1118),
 np.int64(479),
 np.int64(646),
 np.int64(1578),
 np.int64(527),
 np.int64(1225),
 np.int64(345),
 np.int64(292),
 np.int64(896),
 np.int64(372),
 np.int64(428),
 np.int64(1136),
 np.int64(1131),
 np.int64(1084),
 np.int64(845),
 np.int64(814),
 np

## Ways to improve metrics:

Hard negative sampling instead of random sampling

1:4 pos:neg ratio instead of 1:1

Better MLP structure

More Epochs, tuning ...


In [196]:
print(len(user_disliked_items), "/", num_users, "users have any 1-3 ratings")
print(np.mean([len(v) for v in user_disliked_items.values()]))
# So the problem is not we have too many users with 0 1-3 rating. each user rate avg about 47,4 movies 1-3, and we have 941/943

941 / 943 users have any 1-3 ratings
47.42295430393199


In [158]:
collision_count = 0
total = 0
for _, row in train_df.iterrows():
    total += 1
    # compare pos and disliked-neg genre vectors within same user's rows — 
    # simpler: check overall how many positive/disliked pairs share identical genre vectors
    
# simpler global check:
import numpy as np
genre_signature = [tuple(row) for row in movie_features]
from collections import Counter
sig_counts = Counter(genre_signature)
print("unique genre signatures:", len(sig_counts))
print("total movies:", len(movie_features))
print("most common signature count:", sig_counts.most_common(5))

unique genre signatures: 693
total movies: 1682
most common signature count: [((np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.4640385880630426)), 95), ((np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.5342349101353536)), 57), ((np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.floa

In [159]:
print(user_features.shape)  # sanity check dims
print(len(np.unique(user_features, axis=0)), "/", num_users, "unique user vectors")

(943, 34)
841 / 943 unique user vectors


### We add a hard negative (1-3 ratings) but why does the model do worse?

## We spot the culprit, it's because of the similarity in User side and Movie side.
### There are 943 users but only 526 unique embeddings since many users have no difference at all and just identical
### 1682 movies but only 216 unique embeddings since many movies have no difference and just identical

### This make the model worse since it confused the model. Let's say we have a movie 1 that is rated 5 by user 10 and its only property is "drama", another movie 2 rated 1 by user 10 and its only property is "drama". Since both movies have the exact same property, which is drama and only drama, their embeddings are now 1 embedding. The "combined" embedding tell the model to push it toward a high score since movie 1 rated 5 but at the same time tell the model to push it toward 0 since movie 2 rated 1. That's contradictory

### In conclusion, 1 way to solve this is adding distinctive features. Such as zipcode for user and release date for movie. Even though adding IDs will completely solve the identical problem ,we don't want to add UserID and MovieID since this addition will bring our model to become a hybrid CF+CBF

In [ ]:

# TODO : add one-hot zipcode (region only not exact zipcode) to user_features for info richness
# TODO : make combination genre + release year for even more distinctive info. 1990 drama is different from 2020 drama for exmaple